# Экзотические сортировки: Бармен, Гном, Расчёска

Три алгоритма, которые редко встречаются в учебниках, но имеют любопытные свойства.

In [ ]:
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (13, 6)
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

SIZES = [2**k for k in range(1, 14)]  # 2 .. 8192

---
## 1. Сортировка бармена (Cocktail Shaker Sort)

Двунаправленная пузырьковая сортировка: проход слева направо «всплывает» максимум,
затем проход справа налево «утапливает» минимум. Границы сужаются с обеих сторон.

- **Худший случай**: $\Theta(n^2)$
- **Лучший случай** (уже отсортировано): $\Theta(n)$ — один проход без обменов
- **Средний случай**: $\Theta(n^2)$

Преимущество перед обычным Bubble Sort: быстрее справляется с «черепахами» — маленькими элементами в конце массива.

In [ ]:
def cocktail_shaker_sort(data):
    arr = list(data)
    n = len(arr)
    comparisons = 0
    left = 0
    right = n - 1
    swapped = True

    while swapped:
        swapped = False
        for i in range(left, right):
            comparisons += 1
            if arr[i] > arr[i + 1]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                swapped = True
        right -= 1

        if not swapped:
            break

        swapped = False
        for i in range(right, left, -1):
            comparisons += 1
            if arr[i] < arr[i - 1]:
                arr[i], arr[i - 1] = arr[i - 1], arr[i]
                swapped = True
        left += 1

    return arr, comparisons


# Быстрая проверка
test = [5, 3, 8, 1, 2]
sorted_arr, c = cocktail_shaker_sort(test)
print(f'Вход: {test}')
print(f'Выход: {sorted_arr}, сравнений: {c}')

---
## 2. Сортировка гномов (Gnome Sort)

Садовый гном расставляет цветочные горшки: идёт вперёд, если текущий горшок на месте,
и отступает назад, меняя горшки местами, если порядок нарушен.
Дойдя до конца, гном заканчивает работу.

По сути — Insertion Sort без вложенного цикла (один `while`).

- **Худший случай**: $\Theta(n^2)$
- **Лучший случай**: $\Theta(n)$
- **Средний случай**: $\Theta(n^2)$

In [ ]:
def gnome_sort(data):
    arr = list(data)
    n = len(arr)
    comparisons = 0
    i = 1

    while i < n:
        comparisons += 1
        if arr[i] >= arr[i - 1]:
            i += 1
        else:
            arr[i], arr[i - 1] = arr[i - 1], arr[i]
            if i > 1:
                i -= 1
            else:
                i += 1

    return arr, comparisons


test = [5, 3, 8, 1, 2]
sorted_arr, c = gnome_sort(test)
print(f'Вход: {test}')
print(f'Выход: {sorted_arr}, сравнений: {c}')

---
## 3. Сортировка расчёской (Comb Sort)

Улучшение Bubble Sort: вместо сравнения соседей берётся **зазор** (gap),
который на каждом проходе уменьшается в $\approx 1.3$ раза (фактор сжатия Добосевича).
Когда gap = 1 — обычный проход пузырьком до полной отсортированности.

- **Худший случай**: $\Theta(n^2 / 2^p)$ для определённых последовательностей зазоров; в среднем ведёт себя как $O(n \log n)$ / $O(n^2 / 2^p)$
- **Средний случай**: эмпирически $\approx O(n \log n)$
- Фактор сжатия 1.2473… теоретически оптимален (Dobosiewicz, 1980; Lacey & Box, 1991)

In [ ]:
def comb_sort(data, shrink_factor=1.3):
    arr = list(data)
    n = len(arr)
    comparisons = 0
    gap = n
    sorted_flag = False

    while not sorted_flag:
        gap = max(1, int(gap / shrink_factor))
        sorted_flag = (gap == 1)

        for i in range(n - gap):
            comparisons += 1
            if arr[i] > arr[i + gap]:
                arr[i], arr[i + gap] = arr[i + gap], arr[i]
                sorted_flag = False

    return arr, comparisons


test = [5, 3, 8, 1, 2]
sorted_arr, c = comb_sort(test)
print(f'Вход: {test}')
print(f'Выход: {sorted_arr}, сравнений: {c}')

---
## 4. Эксперименты

Запуск на случайных массивах размера $n = 2^1, \ldots, 2^{13}$ + худший случай (обратная сортировка).

In [ ]:
random.seed(42)
np.random.seed(42)

results = {
    'cocktail_rand':  [],
    'cocktail_worst': [],
    'gnome_rand':     [],
    'gnome_worst':    [],
    'comb_rand':      [],
    'comb_worst':     [],
}

# Теоретические кривые для сравнения
theory_n2 = []
theory_nlogn = []

for n in SIZES:
    data_random  = list(np.random.permutation(n))
    data_reverse = list(range(n - 1, -1, -1))

    _, c = cocktail_shaker_sort(data_random)
    results['cocktail_rand'].append(c)
    _, c = cocktail_shaker_sort(data_reverse)
    results['cocktail_worst'].append(c)

    _, c = gnome_sort(data_random)
    results['gnome_rand'].append(c)
    _, c = gnome_sort(data_reverse)
    results['gnome_worst'].append(c)

    _, c = comb_sort(data_random)
    results['comb_rand'].append(c)
    _, c = comb_sort(data_reverse)
    results['comb_worst'].append(c)

    theory_n2.append(n * (n - 1) / 2)
    theory_nlogn.append(n * math.log2(n) if n > 1 else 0)

    print(f'n = {n:>6d}  done')

print('\nГотово.')

---
## 5. Графики

### 5.1 Каждый алгоритм: случайные vs обратно отсортированные данные

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

configs = [
    ('Cocktail Shaker Sort (Бармен)', 'cocktail_rand', 'cocktail_worst'),
    ('Gnome Sort (Гном)',             'gnome_rand',    'gnome_worst'),
    ('Comb Sort (Расчёска)',          'comb_rand',     'comb_worst'),
]

for ax, (title, key_rand, key_worst) in zip(axes, configs):
    ax.plot(SIZES, results[key_rand],  'o-', label='Случайные данные', markersize=4)
    ax.plot(SIZES, results[key_worst], 's-', label='Обратный порядок', markersize=4)
    ax.plot(SIZES, theory_n2, '--', color='gray', alpha=0.5, label=r'$n(n{-}1)/2$')
    ax.set_xlabel('n')
    ax.set_ylabel('Сравнения')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Сводное сравнение (случайные данные, лог-лог)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(SIZES, results['cocktail_rand'], 'o-',  label='Cocktail Shaker (Бармен)', markersize=5)
ax.plot(SIZES, results['gnome_rand'],    's-',  label='Gnome Sort (Гном)', markersize=5)
ax.plot(SIZES, results['comb_rand'],     '^-',  label='Comb Sort (Расчёска)', markersize=5)
ax.plot(SIZES, theory_n2,    '--', color='gray',   alpha=0.6, label=r'$n(n{-}1)/2$ (квадратичная)')
ax.plot(SIZES, theory_nlogn, '--', color='green',  alpha=0.6, label=r'$n \log_2 n$')

ax.set_xscale('log', base=2)
ax.set_yscale('log', base=2)
ax.set_xlabel('n', fontsize=13)
ax.set_ylabel('Число сравнений', fontsize=13)
ax.set_title('Бармен vs Гном vs Расчёска (случайные данные)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Отношение к $n^2$: насколько далеко от квадратичности?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for label, key, marker in [
    ('Cocktail Shaker', 'cocktail_rand', 'o'),
    ('Gnome Sort',      'gnome_rand',    's'),
    ('Comb Sort',       'comb_rand',     '^'),
]:
    ratios = [c / (n * n) for c, n in zip(results[key], SIZES)]
    ax.plot(SIZES, ratios, f'{marker}-', label=label, markersize=5)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_xlabel('n', fontsize=13)
ax.set_ylabel(r'Сравнения / $n^2$', fontsize=13)
ax.set_title(r'Нормированное число сравнений (к $n^2$), случайные данные', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Таблица результатов

In [ ]:
print(f'{"n":>6s} | {"Бармен (случ)":>14s} {"(обр)":>10s} | '
      f'{"Гном (случ)":>12s} {"(обр)":>10s} | '
      f'{"Расчёска (случ)":>16s} {"(обр)":>10s} | '
      f'{"n(n-1)/2":>10s}')
print('-' * 105)

for i, n in enumerate(SIZES):
    print(f'{n:>6d} | '
          f'{results["cocktail_rand"][i]:>14d} {results["cocktail_worst"][i]:>10d} | '
          f'{results["gnome_rand"][i]:>12d} {results["gnome_worst"][i]:>10d} | '
          f'{results["comb_rand"][i]:>16d} {results["comb_worst"][i]:>10d} | '
          f'{int(theory_n2[i]):>10d}')

---
## 7. Выводы

| Алгоритм | Худший | Средний | Лучший | Особенность |
|----------|--------|---------|--------|-------------|
| **Cocktail Shaker** | $\Theta(n^2)$ | $\Theta(n^2)$ | $\Theta(n)$ | Двусторонний Bubble Sort; убирает «черепах» |
| **Gnome Sort** | $\Theta(n^2)$ | $\Theta(n^2)$ | $\Theta(n)$ | Insertion Sort в один `while`-цикл |
| **Comb Sort** | $\Theta(n^2)$ | $\approx O(n \log n)$ | $\Theta(n \log n)$ | Gap-стратегия; на практике близок к $n \log n$ |

**Comb Sort** — единственный из трёх, который на случайных данных ведёт себя **существенно лучше** квадратичного:
на графике 5.2 его кривая растёт вместе с $n \log n$, а не с $n^2$.
Бармен и Гном — по сути перестановки Bubble/Insertion Sort и остаются квадратичными.